# MiWay GTFS Route Efficiency Project - Notebook 4

## Frequency and service availability metrics

This notebook measures how much service each MiWay route provides on the representative weekday: **Tuesday, May 5, 2026**.

It calculates:

1. daily trips by route and direction,
2. first and last scheduled departures,
3. service span,
4. average, median, and high-end headways,
5. AM peak, midday, PM peak, evening, and late-night trip counts,
6. directional balance.

These metrics describe availability: how often buses come, how long service runs, and whether both directions are served evenly.

## 1. Import libraries

In [ ]:
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 150)

try:
    import matplotlib.pyplot as plt
    plt.style.use('seaborn-v0_8-whitegrid')
    HAS_MATPLOTLIB = True
except ImportError:
    HAS_MATPLOTLIB = False
    print('matplotlib is not installed, so chart cells will be skipped.')

## 2. Locate project folders

This notebook works from either the project root or the `Notebooks/` folder.

In [ ]:
candidate_roots = [Path.cwd(), Path.cwd().parent]

PROJECT_ROOT = next(
    (root for root in candidate_roots if (root / 'Data' / 'google_transit.zip').exists()),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not find Data/google_transit.zip from this notebook location.')

DATA_DIR = PROJECT_ROOT / 'Data'
OUTPUT_DIR = PROJECT_ROOT / 'Outputs'
ZIP_PATH = DATA_DIR / 'google_transit.zip'

OUTPUT_DIR.mkdir(exist_ok=True)

print(f'Project root: {PROJECT_ROOT.resolve()}')
print(f'GTFS ZIP:     {ZIP_PATH.resolve()}')
print(f'Outputs:      {OUTPUT_DIR.resolve()}')

## 3. Load trip-level schedule data

Notebook 3 already created a clean trip-level table with parsed departure times. If that file exists, we use it. If not, this notebook rebuilds the required fields directly from the GTFS ZIP.

In [ ]:
ANALYSIS_DATE = pd.Timestamp('2026-05-05')
ANALYSIS_DATE_INT = int(ANALYSIS_DATE.strftime('%Y%m%d'))

TRIP_SPEED_PATH = OUTPUT_DIR / 'weekday_trip_speed_metrics_2026-05-05.csv'


def read_gtfs_table(zip_path: Path, filename: str) -> pd.DataFrame:
    """Read one GTFS text table directly from the ZIP archive."""
    with zipfile.ZipFile(zip_path, 'r') as zf:
        if filename not in zf.namelist():
            raise FileNotFoundError(f'{filename} is missing from {zip_path.name}')
        with zf.open(filename) as file:
            return pd.read_csv(file)


def gtfs_time_to_seconds(value):
    """Convert a GTFS HH:MM:SS time string to seconds after the service day starts."""
    if pd.isna(value):
        return np.nan
    hours, minutes, seconds = str(value).split(':')
    return int(hours) * 3600 + int(minutes) * 60 + int(seconds)


if TRIP_SPEED_PATH.exists():
    weekday_trip_schedule = pd.read_csv(TRIP_SPEED_PATH)
    print(f'Loaded trip-level schedule from {TRIP_SPEED_PATH.name}')
else:
    print('Notebook 3 output not found, rebuilding trip-level schedule from GTFS.')

    routes = read_gtfs_table(ZIP_PATH, 'routes.txt')
    trips = read_gtfs_table(ZIP_PATH, 'trips.txt')
    stop_times = read_gtfs_table(ZIP_PATH, 'stop_times.txt')
    calendar_dates = read_gtfs_table(ZIP_PATH, 'calendar_dates.txt')
    feed_info = read_gtfs_table(ZIP_PATH, 'feed_info.txt')

    feed_start = pd.to_datetime(str(feed_info.loc[0, 'feed_start_date']), format='%Y%m%d')
    feed_end = pd.to_datetime(str(feed_info.loc[0, 'feed_end_date']), format='%Y%m%d')
    if not (feed_start <= ANALYSIS_DATE <= feed_end):
        raise ValueError('Analysis date is outside the feed validity period.')

    active_services = calendar_dates.loc[
        (calendar_dates['date'] == ANALYSIS_DATE_INT) &
        (calendar_dates['exception_type'] == 1),
        'service_id'
    ].drop_duplicates()

    weekday_trips = trips.loc[trips['service_id'].isin(active_services)].copy()
    weekday_trips_with_routes = weekday_trips.merge(routes, on='route_id', how='left', validate='many_to_one')

    active_trip_ids = weekday_trips_with_routes['trip_id'].drop_duplicates()
    weekday_stop_times = stop_times.loc[stop_times['trip_id'].isin(active_trip_ids)].copy()
    weekday_stop_times['arrival_seconds'] = weekday_stop_times['arrival_time'].apply(gtfs_time_to_seconds)
    weekday_stop_times['departure_seconds'] = weekday_stop_times['departure_time'].apply(gtfs_time_to_seconds)
    weekday_stop_times = weekday_stop_times.sort_values(['trip_id', 'stop_sequence'])

    trip_time_summary = (
        weekday_stop_times
        .groupby('trip_id', as_index=False)
        .agg(
            first_departure_seconds=('departure_seconds', 'first'),
            last_arrival_seconds=('arrival_seconds', 'last'),
            first_departure_time=('departure_time', 'first'),
            last_arrival_time=('arrival_time', 'last'),
            stop_count=('stop_id', 'count')
        )
    )

    weekday_trip_schedule = weekday_trips_with_routes.merge(
        trip_time_summary,
        on='trip_id',
        how='left',
        validate='one_to_one'
    )

weekday_trip_schedule['start_hour'] = weekday_trip_schedule['first_departure_seconds'] / 3600

print(f'Analysis date: {ANALYSIS_DATE:%A, %B %d, %Y}')
print(f'Trips available for frequency analysis: {len(weekday_trip_schedule):,}')
print(f'Routes available for frequency analysis: {weekday_trip_schedule["route_id"].nunique()}')

## 4. Assign service periods

We group trips into practical planning periods based on their first departure time.

In [ ]:
def assign_service_period(start_hour: float) -> str:
    if pd.isna(start_hour):
        return 'unknown'
    if 0 <= start_hour < 6:
        return 'early morning'
    if 6 <= start_hour < 9:
        return 'AM peak'
    if 9 <= start_hour < 15:
        return 'midday'
    if 15 <= start_hour < 19:
        return 'PM peak'
    if 19 <= start_hour < 22:
        return 'evening'
    if 22 <= start_hour < 30:
        return 'late night'
    return 'other'


period_order = ['early morning', 'AM peak', 'midday', 'PM peak', 'evening', 'late night', 'other', 'unknown']

weekday_trip_schedule['service_period'] = weekday_trip_schedule['start_hour'].apply(assign_service_period)
weekday_trip_schedule['service_period'] = pd.Categorical(
    weekday_trip_schedule['service_period'],
    categories=period_order,
    ordered=True
)

weekday_trip_schedule['departure_clock'] = weekday_trip_schedule['first_departure_seconds'].apply(
    lambda x: f'{int(x // 3600):02d}:{int((x % 3600) // 60):02d}' if pd.notna(x) else np.nan
)

weekday_trip_schedule[['route_short_name', 'trip_id', 'direction_id', 'first_departure_time', 'start_hour', 'service_period']].head(10)

## 5. Direction-level service span

Transit frequency is directional. A route can appear frequent overall while one direction has much less service, so we calculate direction-level metrics first.

In [ ]:
direction_service_summary = (
    weekday_trip_schedule
    .groupby(['route_id', 'route_short_name', 'route_long_name', 'direction_id'], observed=False, as_index=False)
    .agg(
        trips=('trip_id', 'nunique'),
        first_departure_seconds=('first_departure_seconds', 'min'),
        last_departure_seconds=('first_departure_seconds', 'max'),
        first_arrival_seconds=('last_arrival_seconds', 'min'),
        last_arrival_seconds=('last_arrival_seconds', 'max')
    )
)

direction_service_summary['departure_span_hours'] = (
    direction_service_summary['last_departure_seconds'] - direction_service_summary['first_departure_seconds']
) / 3600

direction_service_summary['full_service_span_hours'] = (
    direction_service_summary['last_arrival_seconds'] - direction_service_summary['first_departure_seconds']
) / 3600

direction_service_summary['first_departure'] = direction_service_summary['first_departure_seconds'].apply(
    lambda x: f'{int(x // 3600):02d}:{int((x % 3600) // 60):02d}' if pd.notna(x) else np.nan
)
direction_service_summary['last_departure'] = direction_service_summary['last_departure_seconds'].apply(
    lambda x: f'{int(x // 3600):02d}:{int((x % 3600) // 60):02d}' if pd.notna(x) else np.nan
)

direction_service_summary.head(20)

## 6. Calculate headways between consecutive departures

A headway is the scheduled wait between one bus and the next on the same route and direction. We calculate headways from consecutive first departures.

In [ ]:
departure_events = weekday_trip_schedule[[
    'route_id', 'route_short_name', 'route_long_name', 'direction_id',
    'trip_id', 'first_departure_seconds', 'first_departure_time', 'service_period'
]].dropna(subset=['first_departure_seconds']).copy()

departure_events = departure_events.sort_values(['route_id', 'direction_id', 'first_departure_seconds', 'trip_id'])

departure_events['previous_departure_seconds'] = departure_events.groupby(
    ['route_id', 'direction_id']
)['first_departure_seconds'].shift(1)

departure_events['headway_min'] = (
    departure_events['first_departure_seconds'] - departure_events['previous_departure_seconds']
) / 60

headway_observations = departure_events.dropna(subset=['headway_min']).copy()

headway_observations = headway_observations.loc[
    (headway_observations['headway_min'] > 0) &
    (headway_observations['headway_min'] <= 240)
].copy()

print(f'Headway observations: {len(headway_observations):,}')
headway_observations.head(20)

## 7. Direction-level headway summary

Median headway is often more stable than average headway, because a few late-night gaps can pull the average upward.

In [ ]:
direction_headway_summary = (
    headway_observations
    .groupby(['route_id', 'direction_id'], as_index=False)
    .agg(
        avg_headway_min=('headway_min', 'mean'),
        median_headway_min=('headway_min', 'median'),
        p90_headway_min=('headway_min', lambda s: s.quantile(0.90)),
        max_headway_min=('headway_min', 'max'),
        headway_observations=('headway_min', 'count')
    )
)

direction_frequency_summary = direction_service_summary.merge(
    direction_headway_summary,
    on=['route_id', 'direction_id'],
    how='left',
    validate='one_to_one'
)

direction_frequency_summary.head(20)

## 8. Route-level frequency and availability summary

Now we summarize from directions back to routes. Directional balance is calculated as the smaller directional trip count divided by the larger directional trip count. A value close to 1 means both directions have similar service.

In [ ]:
route_service_summary = (
    weekday_trip_schedule
    .groupby(['route_id', 'route_short_name', 'route_long_name'], as_index=False)
    .agg(
        scheduled_trips=('trip_id', 'nunique'),
        directions=('direction_id', 'nunique'),
        first_departure_seconds=('first_departure_seconds', 'min'),
        last_departure_seconds=('first_departure_seconds', 'max'),
        last_arrival_seconds=('last_arrival_seconds', 'max')
    )
)

route_service_summary['departure_span_hours'] = (
    route_service_summary['last_departure_seconds'] - route_service_summary['first_departure_seconds']
) / 3600
route_service_summary['full_service_span_hours'] = (
    route_service_summary['last_arrival_seconds'] - route_service_summary['first_departure_seconds']
) / 3600

route_headway_summary = (
    headway_observations
    .groupby('route_id', as_index=False)
    .agg(
        avg_headway_min=('headway_min', 'mean'),
        median_headway_min=('headway_min', 'median'),
        p90_headway_min=('headway_min', lambda s: s.quantile(0.90)),
        max_headway_min=('headway_min', 'max'),
        headway_observations=('headway_min', 'count')
    )
)

direction_trip_counts = (
    weekday_trip_schedule
    .groupby(['route_id', 'direction_id'], as_index=False)
    .agg(direction_trips=('trip_id', 'nunique'))
)

direction_balance = (
    direction_trip_counts
    .groupby('route_id', as_index=False)
    .agg(
        min_direction_trips=('direction_trips', 'min'),
        max_direction_trips=('direction_trips', 'max')
    )
)
direction_balance['directional_balance_ratio'] = (
    direction_balance['min_direction_trips'] / direction_balance['max_direction_trips']
)

route_frequency_summary = (
    route_service_summary
    .merge(route_headway_summary, on='route_id', how='left', validate='one_to_one')
    .merge(direction_balance, on='route_id', how='left', validate='one_to_one')
)

route_frequency_summary['first_departure'] = route_frequency_summary['first_departure_seconds'].apply(
    lambda x: f'{int(x // 3600):02d}:{int((x % 3600) // 60):02d}' if pd.notna(x) else np.nan
)
route_frequency_summary['last_departure'] = route_frequency_summary['last_departure_seconds'].apply(
    lambda x: f'{int(x // 3600):02d}:{int((x % 3600) // 60):02d}' if pd.notna(x) else np.nan
)

route_frequency_summary.head(20)

## 9. Trips by route and service period

This table shows when each route's service is concentrated.

In [ ]:
route_period_trips = (
    weekday_trip_schedule
    .groupby(['route_id', 'route_short_name', 'route_long_name', 'service_period'], observed=False, as_index=False)
    .agg(trips=('trip_id', 'nunique'))
)

route_period_pivot = (
    route_period_trips
    .pivot_table(
        index=['route_id', 'route_short_name', 'route_long_name'],
        columns='service_period',
        values='trips',
        fill_value=0,
        observed=False
    )
    .reset_index()
)

route_period_pivot.columns.name = None
route_period_pivot.head(20)

## 10. Headways by service period

Peak and off-peak headways help reveal whether a route is only useful at rush hour or provides all-day service.

In [ ]:
route_period_headways = (
    headway_observations
    .groupby(['route_id', 'route_short_name', 'route_long_name', 'service_period'], observed=False, as_index=False)
    .agg(
        avg_headway_min=('headway_min', 'mean'),
        median_headway_min=('headway_min', 'median'),
        p90_headway_min=('headway_min', lambda s: s.quantile(0.90)),
        headway_observations=('headway_min', 'count')
    )
)

period_headway_pivot = route_period_headways.pivot_table(
    index=['route_id', 'route_short_name', 'route_long_name'],
    columns='service_period',
    values='median_headway_min',
    observed=False
).reset_index()

period_headway_pivot.columns.name = None
period_headway_pivot = period_headway_pivot.rename(columns={
    'early morning': 'median_headway_early_morning_min',
    'AM peak': 'median_headway_am_peak_min',
    'midday': 'median_headway_midday_min',
    'PM peak': 'median_headway_pm_peak_min',
    'evening': 'median_headway_evening_min',
    'late night': 'median_headway_late_night_min',
    'other': 'median_headway_other_min',
    'unknown': 'median_headway_unknown_min'
})

period_headway_pivot.head(20)

## 11. Final route-level frequency table

This combines all route-level availability metrics into one table for Notebook 5.

In [ ]:
period_trip_columns = [col for col in period_order if col in route_period_pivot.columns]
route_period_pivot_renamed = route_period_pivot.rename(columns={
    'early morning': 'trips_early_morning',
    'AM peak': 'trips_am_peak',
    'midday': 'trips_midday',
    'PM peak': 'trips_pm_peak',
    'evening': 'trips_evening',
    'late night': 'trips_late_night',
    'other': 'trips_other',
    'unknown': 'trips_unknown'
})

route_frequency_service_metrics = (
    route_frequency_summary
    .merge(
        route_period_pivot_renamed,
        on=['route_id', 'route_short_name', 'route_long_name'],
        how='left',
        validate='one_to_one'
    )
    .merge(
        period_headway_pivot,
        on=['route_id', 'route_short_name', 'route_long_name'],
        how='left',
        validate='one_to_one'
    )
)

trip_period_cols = [col for col in route_frequency_service_metrics.columns if col.startswith('trips_')]
route_frequency_service_metrics[trip_period_cols] = route_frequency_service_metrics[trip_period_cols].fillna(0).astype(int)

route_frequency_service_metrics['all_day_service_flag'] = np.where(
    (route_frequency_service_metrics.get('trips_am_peak', 0) > 0) &
    (route_frequency_service_metrics.get('trips_midday', 0) > 0) &
    (route_frequency_service_metrics.get('trips_pm_peak', 0) > 0) &
    (route_frequency_service_metrics.get('trips_evening', 0) > 0),
    1,
    0
)

round_cols = [
    'departure_span_hours', 'full_service_span_hours', 'avg_headway_min',
    'median_headway_min', 'p90_headway_min', 'max_headway_min',
    'directional_balance_ratio'
] + [col for col in route_frequency_service_metrics.columns if col.startswith('median_headway_')]

for col in round_cols:
    if col in route_frequency_service_metrics.columns:
        route_frequency_service_metrics[col] = route_frequency_service_metrics[col].round(2)

route_frequency_service_metrics = route_frequency_service_metrics.sort_values(
    ['all_day_service_flag', 'median_headway_min', 'scheduled_trips'],
    ascending=[False, True, False]
)

route_frequency_service_metrics.head(25)

## 12. Most frequent routes

Routes with low median headways and all-day service are usually the most convenient for riders because buses come often across the whole day. We filter out very limited special-purpose routes here so they do not appear frequent just because a few trips are tightly clustered.

In [ ]:
most_frequent_routes = route_frequency_service_metrics.loc[
    (route_frequency_service_metrics['all_day_service_flag'] == 1) &
    (route_frequency_service_metrics['scheduled_trips'] >= 20)
][[
    'route_short_name', 'route_long_name', 'scheduled_trips', 'first_departure',
    'last_departure', 'full_service_span_hours', 'median_headway_min',
    'p90_headway_min', 'directional_balance_ratio'
]].head(15)

most_frequent_routes

## 13. Routes with the longest scheduled gaps

High p90 headway means riders may face long waits at some points in the day.

In [ ]:
longest_gap_routes = route_frequency_service_metrics.sort_values(
    ['p90_headway_min', 'median_headway_min'],
    ascending=[False, False]
)[[
    'route_short_name', 'route_long_name', 'scheduled_trips', 'median_headway_min',
    'p90_headway_min', 'max_headway_min', 'full_service_span_hours', 'all_day_service_flag'
]].head(15)

longest_gap_routes

## 14. Routes with uneven directional service

A low directional balance ratio can indicate branch patterns, school trips, commuter-oriented service, or uneven bidirectional usefulness.

In [ ]:
least_balanced_routes = route_frequency_service_metrics.sort_values(
    'directional_balance_ratio',
    ascending=True
)[[
    'route_short_name', 'route_long_name', 'scheduled_trips', 'min_direction_trips',
    'max_direction_trips', 'directional_balance_ratio', 'directions'
]].head(15)

least_balanced_routes

## 15. Optional charts

These charts render when `matplotlib` is installed. The rest of the notebook does not depend on it.

In [ ]:
if HAS_MATPLOTLIB:
    fig, ax = plt.subplots(figsize=(10, 6))
    plot_data = most_frequent_routes.sort_values('median_headway_min', ascending=False)
    ax.barh(
        plot_data['route_short_name'].astype(str) + ' - ' + plot_data['route_long_name'].astype(str),
        plot_data['median_headway_min'],
        color='#2a9d8f'
    )
    ax.set_title('Most frequent MiWay routes by median scheduled headway')
    ax.set_xlabel('Median headway (minutes)')
    ax.set_ylabel('Route')
    plt.tight_layout()
else:
    print('Install matplotlib to render this chart.')

In [ ]:
if HAS_MATPLOTLIB:
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.scatter(
        route_frequency_service_metrics['scheduled_trips'],
        route_frequency_service_metrics['median_headway_min'],
        alpha=0.75,
        color='#1f77b4'
    )
    ax.set_title('Daily trips versus median scheduled headway')
    ax.set_xlabel('Scheduled trips')
    ax.set_ylabel('Median headway (minutes)')
    plt.tight_layout()
else:
    print('Install matplotlib to render this chart.')

## 16. Sanity checks

In [ ]:
sanity_checks = {
    'trips in frequency analysis': len(weekday_trip_schedule),
    'routes in frequency summary': len(route_frequency_service_metrics),
    'direction rows': len(direction_frequency_summary),
    'headway observations': len(headway_observations),
    'routes missing median headway': route_frequency_service_metrics['median_headway_min'].isna().sum(),
    'routes with all-day service flag': int(route_frequency_service_metrics['all_day_service_flag'].sum()),
    'minimum median headway': route_frequency_service_metrics['median_headway_min'].min(),
    'maximum median headway': route_frequency_service_metrics['median_headway_min'].max(),
}

pd.Series(sanity_checks, name='value').to_frame()

## 17. Export Notebook 4 outputs

In [ ]:
route_frequency_path = OUTPUT_DIR / 'route_frequency_service_metrics_2026-05-05.csv'
direction_frequency_path = OUTPUT_DIR / 'direction_frequency_service_metrics_2026-05-05.csv'
route_period_trips_path = OUTPUT_DIR / 'route_period_trip_counts_2026-05-05.csv'
route_period_headways_path = OUTPUT_DIR / 'route_period_headways_2026-05-05.csv'
headway_observations_path = OUTPUT_DIR / 'headway_observations_2026-05-05.csv'

route_frequency_service_metrics.to_csv(route_frequency_path, index=False)
direction_frequency_summary.to_csv(direction_frequency_path, index=False)
route_period_trips.to_csv(route_period_trips_path, index=False)
route_period_headways.to_csv(route_period_headways_path, index=False)
headway_observations.to_csv(headway_observations_path, index=False)

print('Exported:')
print('-', route_frequency_path)
print('-', direction_frequency_path)
print('-', route_period_trips_path)
print('-', route_period_headways_path)
print('-', headway_observations_path)

# What this notebook accomplished

Notebook 4 created service availability metrics for each active MiWay route:

- daily scheduled trips,
- first and last departure,
- service span,
- median and high-end headways,
- period-by-period trip counts,
- period-by-period headways,
- directional balance,
- and an all-day service flag.

## Next notebook

Notebook 5 should combine the outputs from Notebooks 2, 3, and 4 into a final route efficiency ranking. That ranking can balance:

1. directness,
2. stop density,
3. scheduled speed,
4. frequency,
5. service span,
6. and directional balance.